# 🧠 ColonoMind — LIMUC Train & Deploy (end-to-end, runnable)

This notebook is **self-contained**. Run it top to bottom and it will:

1. **Download** the public **LIMUC** ulcerative-colitis dataset (Mayo 0–3) from Zenodo.
2. **Train all 5 hybrid models** (ResNet-50, DenseNet-121, EfficientNet-B4, ConvNeXt-Tiny, ViT-B/16) for just **1–2 epochs** to produce weights.
3. **Save loss/accuracy** train-vs-validation curves as images.
4. **Classify** test images and **save the predictions** (CSV + a labelled montage).
5. **Generate the Streamlit app** (`streamlit_colonomind_agent/app.py`) and launch it so you can open it in a browser tab.

> **Dataset:** LIMUC, Polat et al., Zenodo record `5827695`, CC-BY-4.0.
> **Note:** 1–2 epochs on a subset only proves the pipeline works — it is **not** a clinically meaningful model. Increase `EPOCHS` / `MAX_PER_CLASS_*` for real training. A GPU runtime is strongly recommended (ViT-B/16 is slow on CPU).

In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
!pip install -q streamlit joblib lightgbm umap-learn PyWavelets scikit-image \
               opencv-python-headless pillow tensorflow transformers imbalanced-learn tqdm
# Optional public-URL helper for Colab (Section 9):
# !pip install -q pyngrok

In [ ]:
# ============================================================
# CELL 2 — Imports & configuration
# ============================================================
import os, json, zipfile, urllib.request, shutil, time
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import cv2, pywt, scipy.stats, joblib
from pathlib import Path
import matplotlib.pyplot as plt

from skimage.feature import graycomatrix, graycoprops
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, cohen_kappa_score)
from imblearn.over_sampling import SMOTE
import umap

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, GlobalAveragePooling2D,
                                      BatchNormalization, Dropout, Concatenate, Lambda)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

# reproducibility + GPU memory growth
np.random.seed(42); tf.random.set_seed(42)
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e: print(e)
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices('GPU'))

# ---------------- knobs you can change ----------------
IMG_SIZE   = (224, 224)
WAVELET    = 'db1'
EPOCHS     = 2          # 1–2 is enough just to save weights
BATCH_SIZE = 16
LR         = 1e-4
# Cap images per class to keep the demo fast. Set to None to use the FULL dataset.
MAX_PER_CLASS_TRAIN = 300
MAX_PER_CLASS_TEST  = 80
IGNORE_KEYWORDS = ['augment', 'mask', 'seg', '._', 'crop']

# ---------------- paths ----------------
BASE_DIR    = Path.cwd()
# Works whether you run this notebook from the repo root OR from inside the
# streamlit_colonomind_agent/ folder (where it now lives next to requirements.txt).
if BASE_DIR.name == "streamlit_colonomind_agent":
    DEPLOY_DIR = BASE_DIR
else:
    DEPLOY_DIR = BASE_DIR / "streamlit_colonomind_agent"
DATASET_DIR = DEPLOY_DIR / "Dataset" / "LIMUC"
WEIGHTS_DIR = DEPLOY_DIR / "weights"
PLOTS_DIR   = BASE_DIR / "plots"
OUT_DIR     = BASE_DIR / "outputs"
for d in (DATASET_DIR, DEPLOY_DIR, WEIGHTS_DIR, PLOTS_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Working dir:", BASE_DIR)

In [ ]:
# ============================================================
# CELL 3 — Download & extract the LIMUC dataset from Zenodo
# ============================================================
ZENODO = "https://zenodo.org/records/5827695/files/{name}?download=1"
ZIPS = ["train_and_validation_sets.zip", "test_set.zip"]

def _download(url, dest):
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  ✔ already downloaded: {dest.name} ({dest.stat().st_size/1e6:.0f} MB)")
        return
    print(f"  ↓ downloading {dest.name} ...")
    t0 = time.time()
    def hook(b, bsize, total):
        if total > 0 and b % 500 == 0:
            pct = min(100, 100*b*bsize/total)
            print(f"    {pct:5.1f}%  ({b*bsize/1e6:6.0f}/{total/1e6:.0f} MB)", end="\r")
    urllib.request.urlretrieve(url, dest, reporthook=hook)
    print(f"\n  ✔ done in {time.time()-t0:.0f}s")

for name in ZIPS:
    zpath = DATASET_DIR / name
    _download(ZENODO.format(name=name), zpath)
    folder = DATASET_DIR / name.replace(".zip", "")
    if not folder.exists():
        print(f"  ⇲ extracting {name} ...")
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(folder)
    print(f"  ✔ extracted -> {folder}")

# --- locate the actual folders that contain the class subfolders ---
def find_class_root(root):
    root = Path(root)
    # a class root is a dir whose children are class folders holding images
    for p in [root] + [d for d in root.rglob('*') if d.is_dir()]:
        subs = [c for c in p.iterdir() if c.is_dir()]
        if subs and any(any(f.suffix.lower() in ('.bmp','.jpg','.jpeg','.png','.tif')
                            for f in s.iterdir() if f.is_file()) for s in subs):
            return p
    return root

TRAIN_DIR = find_class_root(DATASET_DIR / "train_and_validation_sets")
TEST_DIR  = find_class_root(DATASET_DIR / "test_set")
CLASS_NAMES = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
print("\nTRAIN_DIR :", TRAIN_DIR)
print("TEST_DIR  :", TEST_DIR)
print("CLASS_NAMES:", CLASS_NAMES)
assert len(CLASS_NAMES) >= 2, "Could not detect class folders — check the extraction."

# Human-readable Mayo descriptions, keyed by the detected class-folder names
_MAYO = {0:"normal or inactive disease (intact vascular pattern, no friability).",
         1:"mild activity (erythema, decreased vascular pattern, mild friability).",
         2:"moderate activity (marked erythema, absent vascular pattern, friability, erosions).",
         3:"severe activity (spontaneous bleeding, ulceration)."}
def _describe(name):
    digit = next((int(ch) for ch in name if ch.isdigit()), None)
    base = _MAYO.get(digit, "")
    return f"Mayo Endoscopic Subscore {digit} — {base}" if digit is not None else name
CLASS_DESCRIPTIONS = {c: _describe(c) for c in CLASS_NAMES}
CLASS_DESCRIPTIONS

In [ ]:
# ============================================================
# CELL 4 — Handcrafted features (17 DWT + 3 GLCM = 20) & data loader
#          (identical to the comparison notebook)
# ============================================================
def extract_wavelet_stats(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    LL, (LH, HL, HH) = pywt.dwt2(gray, WAVELET)
    def stats(sb):
        sb_abs = np.abs(sb.flatten()) + 1e-6
        return [np.mean(sb), np.std(sb), np.var(sb), scipy.stats.entropy(sb_abs)]
    feats = stats(LL) + stats(LH) + stats(HL) + stats(HH)   # 16
    feats.append(np.sum(np.square(HH)))                     # + HH energy -> 17
    return feats

def extract_glcm_features(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    glcm = graycomatrix(gray, distances=[1,3,5],
                        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=256, symmetric=True, normed=True)
    return [np.mean(graycoprops(glcm,'contrast')),
            np.mean(graycoprops(glcm,'homogeneity')),
            np.mean(graycoprops(glcm,'dissimilarity'))]     # 3

def extract_combined_features(image):
    feats = extract_wavelet_stats(image) + extract_glcm_features(image)
    assert len(feats) == 20, f"expected 20, got {len(feats)}"
    return feats

def load_dataset(dataset_dir, max_per_class=None):
    images, features, labels, paths = [], [], [], []
    for cls in CLASS_NAMES:
        cdir = Path(dataset_dir) / cls
        if not cdir.exists(): continue
        files = [f for f in sorted(cdir.iterdir())
                 if f.is_file() and not any(k in f.name.lower() for k in IGNORE_KEYWORDS)]
        if max_per_class: files = files[:max_per_class]
        for fp in files:
            img = cv2.imread(str(fp))
            if img is None: continue
            img = cv2.resize(img, IMG_SIZE)
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(rgb); features.append(extract_combined_features(rgb))
            labels.append(cls); paths.append(str(fp))
    return np.array(images), np.array(features), np.array(labels), paths

print("Loading training data...")
Xi_tr, Xf_tr, y_tr_lbl, _        = load_dataset(TRAIN_DIR, MAX_PER_CLASS_TRAIN)
print("Loading test data...")
Xi_te, Xf_te, y_te_lbl, te_paths = load_dataset(TEST_DIR,  MAX_PER_CLASS_TEST)
print("train:", Xi_tr.shape, "| test:", Xi_te.shape)

In [ ]:
# ============================================================
# CELL 5 — Preprocess: normalise, encode, scale, SMOTE, UMAP
# ============================================================
Xi_tr = Xi_tr.astype(np.float32)/255.0
Xi_te = Xi_te.astype(np.float32)/255.0

le = LabelEncoder()
y_tr = le.fit_transform(y_tr_lbl)
y_te = le.transform(y_te_lbl)
NUM_CLASSES = len(le.classes_)

scaler = StandardScaler()
Xf_tr_s = scaler.fit_transform(Xf_tr)
Xf_te_s = scaler.transform(Xf_te)

print("Applying SMOTE...")
smote = SMOTE(random_state=42)
Xf_tr_bal, y_tr_bal = smote.fit_resample(Xf_tr_s, y_tr)
# map synthetic feature rows back to their nearest real image
Xi_tr_bal = []
for feat, lab in zip(Xf_tr_bal, y_tr_bal):
    d = np.linalg.norm(Xf_tr_s[y_tr==lab]-feat, axis=1)
    Xi_tr_bal.append(Xi_tr[y_tr==lab][np.argmin(d)])
Xi_tr_bal = np.array(Xi_tr_bal, dtype=np.float32)

print("Fitting UMAP...")
umap_reducer = umap.UMAP(n_neighbors=10, min_dist=0.05, n_components=2, random_state=42)
Xtr_umap = umap_reducer.fit_transform(Xf_tr_bal)
Xte_umap = umap_reducer.transform(Xf_te_s)

y_tr_cat = to_categorical(y_tr_bal, NUM_CLASSES)
y_te_cat = to_categorical(y_te,     NUM_CLASSES)
train_inputs = [Xi_tr_bal, Xf_tr_bal, Xtr_umap]
val_inputs   = [Xi_te,     Xf_te_s,   Xte_umap]
print("balanced train:", Xi_tr_bal.shape, "| classes:", list(le.classes_))

plt.figure(figsize=(6,5))
plt.scatter(Xtr_umap[:,0], Xtr_umap[:,1], c=y_tr_bal, cmap='viridis', s=8, alpha=.7)
plt.title("UMAP of balanced handcrafted features"); plt.colorbar(label="class")
plt.savefig(PLOTS_DIR/"umap_projection.png", dpi=120, bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
# CELL 6 — Shared architecture + the 5 model branches
# ============================================================
def focal_loss(gamma=2.5, alpha=0.25):
    def loss(yt, yp):
        yp = tf.clip_by_value(yp, 1e-8, 1.0)
        ce = -yt*tf.math.log(yp); w = alpha*tf.math.pow(1-yp, gamma)
        return tf.reduce_mean(tf.reduce_sum(w*ce, axis=1))
    return loss

def _head(base_out, dr):
    x = GlobalAveragePooling2D()(base_out)
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x); x = Dropout(dr)(x)
    return x

def branch_resnet50(shape, dr=0.5):
    from tensorflow.keras.applications import ResNet50
    inp = Input(shape=shape, name='image_input_cnn')
    base = ResNet50(weights='imagenet', include_top=False, input_tensor=inp)
    return Model(inp, _head(base.output, dr), name="ResNet50_Branch")

def branch_densenet121(shape, dr=0.5):
    from tensorflow.keras.applications import DenseNet121
    inp = Input(shape=shape, name='image_input_cnn')
    base = DenseNet121(weights='imagenet', include_top=False, input_tensor=inp)
    return Model(inp, _head(base.output, dr), name="DenseNet121_Branch")

def branch_efficientnetb4(shape, dr=0.5):
    from tensorflow.keras.applications import EfficientNetB4
    inp = Input(shape=shape, name='image_input_cnn')
    base = EfficientNetB4(weights='imagenet', include_top=False, input_tensor=inp)
    return Model(inp, _head(base.output, dr), name="EfficientNetB4_Branch")

def branch_convnexttiny(shape, dr=0.5):
    from tensorflow.keras.applications import ConvNeXtTiny
    inp = Input(shape=shape, name='image_input_cnn')
    base = ConvNeXtTiny(weights='imagenet', include_top=False, input_tensor=inp)
    return Model(inp, _head(base.output, dr), name="ConvNeXtTiny_Branch")

def branch_vitb16(shape, dr=0.5):
    from transformers import TFViTModel
    inp = Input(shape=shape, name='image_input_vit')
    vit = TFViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
    vit.trainable = False
    out = vit(pixel_values=inp)
    cls = Lambda(lambda t: t[:, 0, :])(out.last_hidden_state)
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(cls)
    x = BatchNormalization()(x); x = Dropout(dr)(x)
    return Model(inp, x, name="ViT_B16_Branch")

def build_hybrid_model(branch_fn, num_classes, dr=0.5):
    ii = Input(shape=(224,224,3), name='image_input')
    xc = branch_fn((224,224,3), dr)(ii)
    xc = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(xc)
    xc = BatchNormalization()(xc); xc = Dropout(dr)(xc)
    fi = Input(shape=(20,), name='feat_input')
    xf = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(fi)
    xf = BatchNormalization()(xf); xf = Dropout(dr)(xf)
    ui = Input(shape=(2,), name='umap_input')
    xu = Dense(32, activation='relu', kernel_regularizer=l2(0.01))(ui)
    xu = BatchNormalization()(xu); xu = Dropout(dr)(xu)
    c = Concatenate()([xc, xf, xu])
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(c); x = Dropout(dr)(x)
    out = Dense(num_classes, activation='softmax', name='hybrid_output')(x)
    return Model([ii, fi, ui], out)

MODELS = {
    "ResNet-50":       {"key": "resnet50",       "branch": branch_resnet50},
    "DenseNet-121":    {"key": "densenet121",    "branch": branch_densenet121},
    "EfficientNet-B4": {"key": "efficientnetb4", "branch": branch_efficientnetb4},
    "ConvNeXt-Tiny":   {"key": "convnexttiny",   "branch": branch_convnexttiny},
    "ViT-B-16":        {"key": "vitb16",         "branch": branch_vitb16},
}
print("Models to train:", list(MODELS))

In [ ]:
# ============================================================
# CELL 7 — Train all 5 models (1–2 epochs) & save weights
# ============================================================
all_results, histories, trained = {}, {}, {}

for name, cfg in MODELS.items():
    print(f"\n{'='*60}\n🚀 {name}\n{'='*60}")
    try:
        tf.keras.backend.clear_session()
        model = build_hybrid_model(cfg["branch"], NUM_CLASSES)
        model.compile(optimizer=Adam(LR), loss=focal_loss(2.5, 0.25), metrics=['accuracy'])
        hist = model.fit(train_inputs, y_tr_cat,
                         validation_data=(val_inputs, y_te_cat),
                         batch_size=BATCH_SIZE, epochs=EPOCHS, verbose=2)
        wpath = WEIGHTS_DIR / f"{cfg['key']}.weights.h5"
        model.save_weights(str(wpath))
        proba = model.predict(val_inputs, verbose=0)
        pred = np.argmax(proba, axis=1)
        all_results[name] = {"y_true": y_te, "y_pred": pred, "y_proba": proba}
        histories[name]   = hist.history
        trained[name]     = cfg["key"]
        print(f"✔ {name}: saved weights, test-acc={accuracy_score(y_te, pred):.3f}")
        del model
    except Exception as e:
        print(f"SKIPPED {name}: {type(e).__name__}: {e}")

print("\nTrained OK:", list(trained))

In [ ]:
# ============================================================
# CELL 8 — Save loss & accuracy (train vs validation) curves
# ============================================================
for name, h in histories.items():
    key = MODELS[name]["key"]
    ep = range(1, len(h["loss"])+1)
    fig, ax = plt.subplots(1, 2, figsize=(11,4))
    ax[0].plot(ep, h["loss"], 'o-', label="train")
    ax[0].plot(ep, h["val_loss"], 's-', label="val")
    ax[0].set_title(f"{name} — Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(ep, h["accuracy"], 'o-', label="train")
    ax[1].plot(ep, h["val_accuracy"], 's-', label="val")
    ax[1].set_title(f"{name} — Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR/f"{key}_history.png", dpi=120, bbox_inches='tight')
    plt.show()

# combined validation-accuracy comparison
if histories:
    plt.figure(figsize=(7,4))
    for name, h in histories.items():
        plt.plot(range(1, len(h["val_accuracy"])+1), h["val_accuracy"], 'o-', label=name)
    plt.title("Validation accuracy — all models"); plt.xlabel("epoch"); plt.ylabel("val acc")
    plt.legend(); plt.grid(alpha=.3)
    plt.savefig(PLOTS_DIR/"val_accuracy_comparison.png", dpi=120, bbox_inches='tight'); plt.show()
print("Saved plots to:", PLOTS_DIR)

In [ ]:
# ============================================================
# CELL 9 — Metrics, saved predictions (CSV) & a labelled montage
# ============================================================
metrics_summary = {}
for name, res in all_results.items():
    yt, yp, pr = np.asarray(res["y_true"]), np.asarray(res["y_pred"]), np.asarray(res["y_proba"])
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES)))
    specs=[]
    for i in range(NUM_CLASSES):
        tn = cm.sum()-cm[i,:].sum()-cm[:,i].sum()+cm[i,i]; fp = cm[:,i].sum()-cm[i,i]
        specs.append(tn/(tn+fp+1e-8))
    metrics_summary[name] = {
        "accuracy": float(accuracy_score(yt,yp)),
        "precision_macro": float(precision_score(yt,yp,average='macro',zero_division=0)),
        "recall_macro": float(recall_score(yt,yp,average='macro',zero_division=0)),
        "specificity_macro": float(np.mean(specs)),
        "f1_macro": float(f1_score(yt,yp,average='macro',zero_division=0)),
        "qwk": float(cohen_kappa_score(yt,yp,weights='quadratic')) if len(set(yt))>1 else 0.0,
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(yt,yp,target_names=CLASS_NAMES,
                                                       zero_division=0, output_dict=True),
    }
    # per-model predictions CSV
    df = pd.DataFrame({
        "image": [Path(p).name for p in te_paths],
        "true_label":  [CLASS_NAMES[i] for i in yt],
        "pred_label":  [CLASS_NAMES[i] for i in yp],
        "confidence":  pr.max(axis=1),
    })
    for i,c in enumerate(CLASS_NAMES): df[f"p_{c}"] = pr[:,i]
    df.to_csv(OUT_DIR/f"predictions_{MODELS[name]['key']}.csv", index=False)
print(pd.DataFrame(metrics_summary).T[["accuracy","precision_macro","recall_macro","f1_macro","qwk"]]
      if metrics_summary else "no models trained")

# labelled montage from the best model
if all_results:
    best = max(all_results, key=lambda n: metrics_summary[n]["accuracy"])
    res = all_results[best]; yp = res["y_pred"]; pr = res["y_proba"]
    n = min(12, len(te_paths)); cols=4; rows=int(np.ceil(n/cols))
    plt.figure(figsize=(3*cols, 3*rows))
    for i in range(n):
        img = cv2.cvtColor(cv2.imread(te_paths[i]), cv2.COLOR_BGR2RGB)
        ok = (yp[i]==y_te[i]); plt.subplot(rows,cols,i+1); plt.imshow(img); plt.axis('off')
        plt.title(f"T:{CLASS_NAMES[y_te[i]]}\nP:{CLASS_NAMES[yp[i]]} ({pr[i].max():.2f})",
                  color=('green' if ok else 'red'), fontsize=9)
    plt.suptitle(f"Sample predictions — {best}"); plt.tight_layout()
    plt.savefig(OUT_DIR/"sample_predictions.png", dpi=120, bbox_inches='tight'); plt.show()
    print("Saved predictions + montage to:", OUT_DIR)

In [ ]:
# ============================================================
# CELL 10 — Export preprocessing artifacts + deployment report
#           (weights were already saved in Cell 7)
# ============================================================
joblib.dump({"scaler": scaler, "umap_reducer": umap_reducer, "label_encoder": le,
             "class_names": CLASS_NAMES, "img_size": (224,224), "wavelet": WAVELET},
            DEPLOY_DIR/"preprocess_artifacts.joblib")

model_registry = {name: {"key": trained[name], "weights_path": f"{trained[name]}.weights.h5"}
                  for name in trained}
report = {"project_name": "ColonoMind Diagnostic Agent (LIMUC)",
          "class_names": CLASS_NAMES,
          "class_descriptions": CLASS_DESCRIPTIONS,
          "model_registry": model_registry,
          "metrics_summary": metrics_summary}
json.dump(report, open(DEPLOY_DIR/"deployment_report.json","w"), indent=2)
print("✔ Exported artifacts + deployment_report.json to", DEPLOY_DIR)
print("  models available in app:", list(model_registry))

In [ ]:
# ============================================================
# CELL 11 — Write requirements.txt + README for the deploy folder
# ============================================================
(DEPLOY_DIR/"requirements.txt").write_text('''streamlit==1.37.1
tensorflow-cpu==2.15.1
transformers==4.41.2
numpy==1.26.4
scikit-learn==1.4.2
umap-learn==0.5.6
pandas==2.2.2
scipy==1.13.1
joblib==1.4.2
opencv-python-headless==4.10.0.84
scikit-image==0.23.2
PyWavelets==1.6.0
Pillow==10.4.0
# Optional Claude Q&A:  anthropic==0.34.2
''', encoding="utf-8")

(DEPLOY_DIR/"README.md").write_text('''# ColonoMind Diagnostic Agent (LIMUC)

Streamlit app that classifies a colonoscopy image into Mayo Endoscopic
Subscore classes using one of the trained hybrid models.

## Run locally
    cd streamlit_colonomind_agent
    pip install -r requirements.txt
    streamlit run app.py

## Deploy to Streamlit Community Cloud
- Main file path: streamlit_colonomind_agent/app.py
- The weights/*.h5 files are large (~280 MB each). Use Git LFS or host them
  externally and download at startup rather than committing them.
- Pin tensorflow / scikit-learn / umap-learn / numpy to the versions you
  trained with so the pickled artifacts and weights load cleanly.

Research/educational tool - not a clinical diagnosis.
''', encoding="utf-8")
print("✔ Wrote requirements.txt and README.md")

In [ ]:
# ============================================================
# CELL 12 — Write the Streamlit app (app.py)
# ============================================================
APP_PATH = DEPLOY_DIR / 'app.py'
app_code = r'''
import os
import json
import joblib
import numpy as np
import pandas as pd
import cv2
import pywt
import scipy.stats
import streamlit as st

from pathlib import Path
from PIL import Image
from skimage.feature import graycomatrix, graycoprops

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, GlobalAveragePooling2D, BatchNormalization,
    Dropout, Concatenate, Lambda
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.applications import ResNet50, DenseNet121, EfficientNetB4

try:
    from tensorflow.keras.applications import ConvNeXtTiny
    HAS_CONVNEXT = True
except Exception:
    HAS_CONVNEXT = False

try:
    from transformers import TFViTModel
    HAS_TRANSFORMERS = True
except Exception:
    HAS_TRANSFORMERS = False

# ============================================================
# 0. Page config + paths
# ============================================================
st.set_page_config(
    page_title="ColonoMind Diagnostic Agent",
    page_icon="\U0001F9E0",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.title("\U0001F9E0 ColonoMind Diagnostic Agent")
st.caption(
    "Upload a colonoscopy image, pick a trained model, run the diagnosis, "
    "review the model's report and ask questions. "
    "Research/educational tool — not a clinical diagnosis."
)

BASE_DIR = Path(__file__).resolve().parent
WEIGHTS_DIR = BASE_DIR / "weights"
REPORT_PATH = BASE_DIR / "deployment_report.json"
PREPROCESS_PATH = BASE_DIR / "preprocess_artifacts.joblib"

# Clinical meaning of the Mayo Endoscopic Subscore classes (grounding for Q&A)
MES_INFO = {
    "MES0": "Mayo Endoscopic Subscore 0 — normal or inactive disease "
            "(intact vascular pattern, no friability).",
    "MES1": "Mayo Endoscopic Subscore 1 — mild activity "
            "(erythema, decreased vascular pattern, mild friability).",
    "MES2": "Mayo Endoscopic Subscore 2 — moderate activity "
            "(marked erythema, absent vascular pattern, friability, erosions).",
    "MES3": "Mayo Endoscopic Subscore 3 — severe activity "
            "(spontaneous bleeding, ulceration).",
}

# ============================================================
# 1. Load deployment report + preprocessing artifacts
# ============================================================
@st.cache_data
def load_report():
    with open(REPORT_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

@st.cache_resource
def load_preprocess_artifacts():
    return joblib.load(PREPROCESS_PATH)

report = load_report()
artifacts = load_preprocess_artifacts()

CLASS_NAMES = artifacts["class_names"]
IMG_SIZE = tuple(artifacts["img_size"])
WAVELET = artifacts.get("wavelet", "db1")
scaler = artifacts["scaler"]
umap_reducer = artifacts["umap_reducer"]

MODEL_REGISTRY = report["model_registry"]
METRICS_SUMMARY = report["metrics_summary"]
MES_INFO = report.get("class_descriptions", MES_INFO)

# ============================================================
# 2. Feature extraction  (MUST match the training notebook exactly)
#    17 DWT features (16 stats + HH energy) + 3 GLCM = 20 features
# ============================================================
def extract_wavelet_stats(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    LL, (LH, HL, HH) = pywt.dwt2(gray, WAVELET)

    def stats(subband):
        subband_abs = np.abs(subband.flatten()) + 1e-6
        return [
            float(np.mean(subband)),
            float(np.std(subband)),
            float(np.var(subband)),
            float(scipy.stats.entropy(subband_abs)),
        ]

    dwt_features = stats(LL) + stats(LH) + stats(HL) + stats(HH)   # 16
    dwt_features.append(float(np.sum(np.square(HH))))              # + HH energy -> 17
    return dwt_features

def extract_glcm_features(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    distances = [1, 3, 5]
    angles = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]
    glcm = graycomatrix(gray, distances=distances, angles=angles,
                        levels=256, symmetric=True, normed=True)
    return [
        float(np.mean(graycoprops(glcm, "contrast"))),
        float(np.mean(graycoprops(glcm, "homogeneity"))),
        float(np.mean(graycoprops(glcm, "dissimilarity"))),
    ]                                                             # 3

def extract_combined_features(image_rgb):
    feats = extract_wavelet_stats(image_rgb) + extract_glcm_features(image_rgb)
    assert len(feats) == 20, f"Expected 20 features, got {len(feats)}"
    return np.array(feats, dtype=np.float32)

def preprocess_uploaded_image(uploaded_file):
    pil_img = Image.open(uploaded_file).convert("RGB")
    image_rgb_original = np.array(pil_img)

    image_resized = cv2.resize(image_rgb_original, IMG_SIZE).astype(np.uint8)

    image_input = (image_resized.astype(np.float32) / 255.0)[np.newaxis, ...]

    handcrafted = extract_combined_features(image_resized).reshape(1, -1)
    handcrafted_scaled = scaler.transform(handcrafted)
    umap_feat = umap_reducer.transform(handcrafted_scaled)

    return {
        "pil_image": pil_img,
        "image_input": image_input,
        "handcrafted_scaled": handcrafted_scaled,
        "umap_feat": umap_feat,
    }

# ============================================================
# 3. Model architecture  (identical to the training notebook)
# ============================================================
def _cnn_head(base_model, dropout_rate):
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return x

def create_resnet50_branch(input_shape, dropout_rate=0.5):
    inp = Input(shape=input_shape, name="image_input_cnn")
    base = ResNet50(weights=None, include_top=False, input_tensor=inp)
    return Model(inp, _cnn_head(base, dropout_rate), name="ResNet50_Branch")

def create_densenet121_branch(input_shape, dropout_rate=0.5):
    inp = Input(shape=input_shape, name="image_input_cnn")
    base = DenseNet121(weights=None, include_top=False, input_tensor=inp)
    return Model(inp, _cnn_head(base, dropout_rate), name="DenseNet121_Branch")

def create_efficientnetb4_branch(input_shape, dropout_rate=0.5):
    inp = Input(shape=input_shape, name="image_input_cnn")
    base = EfficientNetB4(weights=None, include_top=False, input_tensor=inp)
    return Model(inp, _cnn_head(base, dropout_rate), name="EfficientNetB4_Branch")

def create_convnexttiny_branch(input_shape, dropout_rate=0.5):
    if not HAS_CONVNEXT:
        raise ImportError("ConvNeXtTiny is not available in this Keras version.")
    inp = Input(shape=input_shape, name="image_input_cnn")
    base = ConvNeXtTiny(weights=None, include_top=False, input_tensor=inp)
    return Model(inp, _cnn_head(base, dropout_rate), name="ConvNeXtTiny_Branch")

def create_vitb16_branch(input_shape, dropout_rate=0.5):
    if not HAS_TRANSFORMERS:
        raise ImportError("transformers is not installed — needed for ViT-B-16.")
    inp = Input(shape=input_shape, name="image_input_vit")
    vit = TFViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
    vit.trainable = False
    outputs = vit(pixel_values=inp)
    cls_token = Lambda(lambda t: t[:, 0, :])(outputs.last_hidden_state)
    x = Dense(512, activation="relu", kernel_regularizer=l2(0.01))(cls_token)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return Model(inp, x, name="ViT_B16_Branch")

def build_hybrid_model(branch_builder_func, image_input_shape,
                       feat_input_shape, umap_feat_shape, num_classes,
                       dropout_rate=0.5):
    image_input = Input(shape=image_input_shape, name="image_input")
    x_cnn = branch_builder_func(image_input_shape, dropout_rate)(image_input)
    x_cnn = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(x_cnn)
    x_cnn = BatchNormalization()(x_cnn)
    x_cnn = Dropout(dropout_rate)(x_cnn)

    feat_input = Input(shape=feat_input_shape, name="feat_input")
    x_feat = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(feat_input)
    x_feat = BatchNormalization()(x_feat)
    x_feat = Dropout(dropout_rate)(x_feat)

    umap_input = Input(shape=umap_feat_shape, name="umap_input")
    x_umap = Dense(32, activation="relu", kernel_regularizer=l2(0.01))(umap_input)
    x_umap = BatchNormalization()(x_umap)
    x_umap = Dropout(dropout_rate)(x_umap)

    combined = Concatenate()([x_cnn, x_feat, x_umap])
    x = Dense(128, activation="relu", kernel_regularizer=l2(0.01))(combined)
    x = Dropout(dropout_rate)(x)
    output = Dense(num_classes, activation="softmax", name="hybrid_output")(x)

    return Model(inputs=[image_input, feat_input, umap_input], outputs=output)

BRANCH_BUILDERS = {
    "ResNet-50": create_resnet50_branch,
    "DenseNet-121": create_densenet121_branch,
    "EfficientNet-B4": create_efficientnetb4_branch,
    "ConvNeXt-Tiny": create_convnexttiny_branch,
    "ViT-B-16": create_vitb16_branch,
}

@st.cache_resource(show_spinner=False)
def load_selected_model(model_name):
    model = build_hybrid_model(
        branch_builder_func=BRANCH_BUILDERS[model_name],
        image_input_shape=(224, 224, 3),
        feat_input_shape=(20,),
        umap_feat_shape=(2,),
        num_classes=len(CLASS_NAMES),
        dropout_rate=0.5,
    )
    weight_path = WEIGHTS_DIR / MODEL_REGISTRY[model_name]["weights_path"]
    if not weight_path.exists():
        raise FileNotFoundError(f"Weight file not found: {weight_path}")
    model.load_weights(str(weight_path))
    return model

# ============================================================
# 4. Sidebar  —  model (weight) dropdown
# ============================================================
st.sidebar.header("⚙️ Model selection")
selected_model_name = st.sidebar.selectbox(
    "Choose which trained weight to run",
    list(MODEL_REGISTRY.keys()),
)
st.sidebar.success(f"Active model: {selected_model_name}")

use_llm = st.sidebar.checkbox("Use Claude for Q&A (optional)", value=False)
api_key_input = ""
if use_llm:
    api_key_input = st.sidebar.text_input(
        "ANTHROPIC_API_KEY", type="password",
        help="Leave empty to use the ANTHROPIC_API_KEY environment variable.",
    )

# ============================================================
# PART 1  —  Upload image  (top)
# ============================================================
st.markdown("---")
st.header("1️⃣ Upload image")

uploaded_file = st.file_uploader(
    "Upload a colonoscopy image",
    type=["png", "jpg", "jpeg", "bmp", "tif", "tiff"],
)

processed = None
if uploaded_file is not None:
    processed = preprocess_uploaded_image(uploaded_file)
    st.image(processed["pil_image"], caption="Uploaded image", width=380)
else:
    st.info("Upload an image to begin.")

# ============================================================
# PART 2  —  Chosen-model report (metrics)  (middle)
# ============================================================
st.markdown("---")
st.header("2️⃣ Model performance report")

if selected_model_name in METRICS_SUMMARY:
    m = METRICS_SUMMARY[selected_model_name]
    c1, c2, c3, c4, c5, c6 = st.columns(6)
    c1.metric("Accuracy", f"{m['accuracy']:.3f}")
    c2.metric("Precision", f"{m['precision_macro']:.3f}")
    c3.metric("Recall/Sens.", f"{m['recall_macro']:.3f}")
    c4.metric("Specificity", f"{m['specificity_macro']:.3f}")
    c5.metric("F1-score", f"{m['f1_macro']:.3f}")
    c6.metric("QWK", f"{m['qwk']:.3f}")

    cm = np.array(m["confusion_matrix"])
    cm_df = pd.DataFrame(
        cm,
        index=[f"True {c}" for c in CLASS_NAMES],
        columns=[f"Pred {c}" for c in CLASS_NAMES],
    )
    st.markdown("**Confusion matrix (validation set)**")
    st.dataframe(cm_df, use_container_width=True)
    st.caption(
        "Metrics reflect this model's evaluation in the training notebook."
    )
else:
    st.warning("No metrics found for this model in deployment_report.json.")

# ============================================================
# PART 3  —  Image + classification result
# ============================================================
st.markdown("---")
st.header("3️⃣ Run diagnosis & result")

if processed is None:
    st.info("Please upload an image first.")
elif st.button("\U0001F50D Run diagnosis", type="primary"):
    with st.spinner(f"Running {selected_model_name}..."):
        model = load_selected_model(selected_model_name)
        y_proba = model.predict(
            [processed["image_input"],
             processed["handcrafted_scaled"],
             processed["umap_feat"]],
            verbose=0,
        )[0]

    pred_idx = int(np.argmax(y_proba))
    pred_class = CLASS_NAMES[pred_idx]
    confidence = float(y_proba[pred_idx])

    st.session_state["last_prediction"] = {
        "model": selected_model_name,
        "predicted_class": pred_class,
        "confidence": confidence,
        "probabilities": {CLASS_NAMES[i]: float(y_proba[i])
                          for i in range(len(CLASS_NAMES))},
    }

# Persist the result across chat interactions
if "last_prediction" in st.session_state:
    pred = st.session_state["last_prediction"]
    left, right = st.columns([1, 1])
    with left:
        if processed is not None:
            st.image(processed["pil_image"], caption="Diagnosed image", width=340)
    with right:
        st.subheader("Classification result")
        st.success(f"Predicted class: **{pred['predicted_class']}**")
        st.metric("Confidence", f"{pred['confidence']:.3f}")
        st.caption(MES_INFO.get(pred["predicted_class"], ""))
        prob_df = pd.DataFrame({
            "Class": list(pred["probabilities"].keys()),
            "Probability": list(pred["probabilities"].values()),
        })
        st.bar_chart(prob_df.set_index("Class"))
        st.caption(f"Prediction produced by: {pred['model']}")

# ============================================================
# PART 4  —  Question & Answer agent  (bottom)
# ============================================================
st.markdown("---")
st.header("4️⃣ Diagnostic Q&A agent")
st.caption(
    "Ask about the selected model's metrics, the MES classes, or the last "
    "prediction. Answers are grounded in the deployment report."
)

if "chat_history" not in st.session_state:
    st.session_state["chat_history"] = []

def build_context():
    return {
        "selected_model": selected_model_name,
        "class_names": CLASS_NAMES,
        "class_descriptions": MES_INFO,
        "selected_model_metrics": METRICS_SUMMARY.get(selected_model_name, {}),
        "last_prediction": st.session_state.get("last_prediction"),
    }

def deterministic_answer(query, ctx):
    q = query.lower()
    m = ctx["selected_model_metrics"]
    pred = ctx["last_prediction"]
    name = ctx["selected_model"]

    metric_keys = [
        ("accuracy", "accuracy", "accuracy"),
        ("precision", "precision_macro", "macro precision"),
        ("specificity", "specificity_macro", "macro specificity"),
        ("f1", "f1_macro", "macro F1-score"),
        ("qwk", "qwk", "quadratic weighted kappa"),
    ]
    for kw, key, label in metric_keys:
        if kw in q:
            val = m.get(key)
            return (f"**{name}** — {label}: "
                    f"{val:.4f}" if isinstance(val, (int, float))
                    else f"{label} is not available for {name}.")
    if "recall" in q or "sensitivity" in q:
        val = m.get("recall_macro")
        return f"**{name}** — macro recall/sensitivity: {val:.4f}" \
               if isinstance(val, (int, float)) else "Recall not available."
    if "confusion" in q:
        return f"Confusion matrix for **{name}**: {m.get('confusion_matrix')}"
    if any(k in q for k in ["mes", "class", "meaning", "severity", "score"]):
        return "\n\n".join(f"- **{k}**: {v}" for k, v in ctx["class_descriptions"].items())
    if any(k in q for k in ["prediction", "result", "diagnos", "predict"]):
        if pred is None:
            return "No prediction yet — upload an image and click **Run diagnosis**."
        desc = ctx["class_descriptions"].get(pred["predicted_class"], "")
        return (f"Latest prediction ({pred['model']}): **{pred['predicted_class']}** "
                f"at {pred['confidence']:.3f} confidence.\n\n{desc}\n\n"
                "This is model output, not a clinical diagnosis.")
    return ("I can answer about this model's accuracy, precision, recall, "
            "specificity, F1, QWK, confusion matrix, the MES classes, and the "
            "latest prediction.")

def claude_answer(query, ctx, api_key):
    try:
        from anthropic import Anthropic
        client = Anthropic(api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"))
        msg = client.messages.create(
            model="claude-sonnet-5",
            max_tokens=600,
            system=(
                "You are a diagnostic assistant for the ColonoMind app. "
                "Answer ONLY from the provided JSON context. Do not invent "
                "metrics or make clinical decisions. State that outputs are "
                "model classifications, not a final diagnosis. Be concise."
            ),
            messages=[{
                "role": "user",
                "content": f"Context JSON:\n{json.dumps(ctx, indent=2)}\n\nQuestion: {query}",
            }],
        )
        return msg.content[0].text
    except Exception as e:
        return deterministic_answer(query, ctx) + f"\n\n_(Claude unavailable: {e})_"

for msg in st.session_state["chat_history"]:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

user_question = st.chat_input("Ask about this model or the prediction...")
if user_question:
    st.session_state["chat_history"].append({"role": "user", "content": user_question})
    with st.chat_message("user"):
        st.markdown(user_question)
    ctx = build_context()
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            if use_llm:
                answer = claude_answer(user_question, ctx, api_key_input)
            else:
                answer = deterministic_answer(user_question, ctx)
        st.markdown(answer)
    st.session_state["chat_history"].append({"role": "assistant", "content": answer})
'''
APP_PATH.write_text(app_code, encoding='utf-8')
print('✔ Streamlit app written to:', APP_PATH)

In [ ]:
# ============================================================
# CELL 13 — Launch the Streamlit app (open it in a browser tab)
# ============================================================
APP_PATH = DEPLOY_DIR / "app.py"

# ---- A) Local machine (Anaconda / terminal) --------------------------------
# This opens a browser tab automatically at http://localhost:8501
#     streamlit run streamlit_colonomind_agent/app.py
# Or launch straight from the notebook (blocks the cell while running):
# !streamlit run streamlit_colonomind_agent/app.py

# ---- B) Google Colab — get a public URL to open in a tab -------------------
# Option 1: ngrok (needs a free token from https://dashboard.ngrok.com)
#   !pip install -q pyngrok
#   from pyngrok import ngrok
#   ngrok.kill()
#   ngrok.set_auth_token("YOUR_NGROK_TOKEN")
#   public_url = ngrok.connect(8501); print("Open this tab:", public_url)
#   !streamlit run streamlit_colonomind_agent/app.py --server.port 8501 --server.headless true &>/content/st.log &
#
# Option 2: localtunnel (no account; password = your Colab public IP)
#   !streamlit run streamlit_colonomind_agent/app.py --server.port 8501 --server.headless true &>/content/st.log &
#   !npx --yes localtunnel --port 8501
#   # print the tunnel password:
#   !curl -s https://loca.lt/mytunnelpassword

print("App path:", APP_PATH)
print("Local:  streamlit run", APP_PATH)
print("Colab:  uncomment section B above to get a public URL.")